In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!apt-get -qq update
!apt-get -qq install mesa-opencl-icd libosmesa6 libosmesa6-dev
!pip -q install smplx pyopengl pyrender trimesh torchgeometry spacepy \
    git+https://github.com/mattloper/chumpy.git

# Clone SPIN and add to path
import sys, os
if not os.path.exists('SPIN'):
    !git clone -q https://github.com/avasenin-14/SPIN
spin_path = os.path.abspath('SPIN')
if spin_path not in sys.path:
    sys.path.append(spin_path)

# Download SPIN pretrained data
%cd SPIN
if not os.path.exists('data/model_checkpoint.pt'):
    !wget -q http://visiondata.cis.upenn.edu/spin/data.tar.gz && tar -xf data.tar.gz && rm data.tar.gz
    !wget -q http://visiondata.cis.upenn.edu/spin/model_checkpoint.pt -P data/
    !wget -q https://github.com/vchoutas/smplify-x/raw/master/smplifyx/prior.py -O smplify/prior.py

# Copy SMPL model files
import shutil
smpl_src = '/content/drive/MyDrive/dissertation/smpl/'
smpl_dst = './data/smpl/'
os.makedirs(smpl_dst, exist_ok=True)
for new_name, orig_name in {
    'SMPL_NEUTRAL.pkl': 'basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl',
    'SMPL_MALE.pkl':    'basicmodel_m_lbs_10_207_0_v1.1.0.pkl',
    'SMPL_FEMALE.pkl':  'basicmodel_f_lbs_10_207_0_v1.1.0.pkl',
}.items():
    src = smpl_src + orig_name
    dst = smpl_dst + new_name
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)

print("Environment ready.")

In [ ]:
import os
os.environ['PYOPENGL_PLATFORM'] = 'egl'

import time
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from torchvision.models import vit_b_16, ViT_B_16_Weights

from models.hmr import HMR, Bottleneck
from models.smpl import SMPL
from utils.geometry import rot6d_to_rotmat
from utils.renderer import Renderer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

SMPL_MEAN_PARAMS = 'data/smpl_mean_params.npz'
CHECKPOINT_PATH  = 'data/model_checkpoint.pt'
FOCAL_LENGTH     = 5000
IMG_RES          = 224

In [ ]:
import tarfile, io
from torch.utils.data import IterableDataset, DataLoader

IMAGES_DIR = '/content/drive/MyDrive/dissertation/humans3.6m/images'
ARCHIVES = [
    f'{IMAGES_DIR}/images.tar.gzaa',
    f'{IMAGES_DIR}/images.tar.gzab',
    f'{IMAGES_DIR}/images.tar.gzac',
    f'{IMAGES_DIR}/images.tar.gzad',
    f'{IMAGES_DIR}/images.tar.gzae',
    f'{IMAGES_DIR}/images.tar.gzaf',
    f'{IMAGES_DIR}/images.tar.gzag',
]

# ViT preprocessing transform
vit_weights = ViT_B_16_Weights.DEFAULT
preprocess  = vit_weights.transforms()


class TarImageDataset(IterableDataset):
    """Stream images from one or more tar archives sequentially."""
    def __init__(self, archive_paths, transform=None):
        if isinstance(archive_paths, str):
            archive_paths = [archive_paths]
        self.archive_paths = archive_paths
        self.transform = transform

    def __iter__(self):
        for path in self.archive_paths:
            print(f"Streaming from {os.path.basename(path)} ...")
            with tarfile.open(path, mode='r') as tar:
                for member in tar:
                    if member.isfile() and member.name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        try:
                            f = tar.extractfile(member)
                            if f is not None:
                                img = Image.open(io.BytesIO(f.read())).convert('RGB')
                                if self.transform:
                                    img = self.transform(img)
                                yield img, member.name
                        except Exception as e:
                            print(f"Skipping {member.name}: {e}")


# Quick-check with first archive only
dataset    = TarImageDataset(ARCHIVES[0], transform=preprocess)
dataloader = DataLoader(dataset, batch_size=4, num_workers=0)

images, names = next(iter(dataloader))
images = images.to(device)
print(f"Batch shape: {images.shape}  |  Files: {names}")

In [ ]:
import json, zipfile, glob as _glob

# 1) H3.6M files already in SPIN data/ directory (from data.tar.gz download)
h36m_files = sorted(_glob.glob('data/*h36m*') + _glob.glob('data/**/*h36m*', recursive=True))
print("══ H3.6M files in SPIN data/ ══")
for f in h36m_files:
    sz = os.path.getsize(f) / 1e6
    print(f"  {f}  ({sz:.1f} MB)")
if not h36m_files:
    print("  (none)")

# 2) Inspect annotations.zip
ANN_ZIP = '/content/drive/MyDrive/dissertation/humans3.6m/annotations.zip'
print(f"\n══ {os.path.basename(ANN_ZIP)} ══")
with zipfile.ZipFile(ANN_ZIP, 'r') as z:
    names = z.namelist()
    exts = {}
    for n in names:
        ext = os.path.splitext(n)[1].lower() or '/'
        exts[ext] = exts.get(ext, 0) + 1
    print(f"  Entries: {len(names)},  Types: {exts}")
    for n in names[:20]:
        print(f"    {n}")

# 3) Bbox / root-joint JSON samples
BBOX_DIR = '/content/drive/MyDrive/dissertation/Bounding box + Root joint coordinate/Human3.6M'
if os.path.isdir(BBOX_DIR):
    for sub in sorted(os.listdir(BBOX_DIR)):
        jp = os.path.join(BBOX_DIR, sub, 'bbox_root_human36m_output.json')
        if os.path.exists(jp):
            with open(jp) as f:
                d = json.load(f)
            sample = d[0] if isinstance(d, list) else next(iter(d.values()))
            print(f"\n══ {sub} ══")
            print(f"  Type: {type(d).__name__}, Entries: {len(d)}, Sample: {sample}")

In [ ]:
spin_model = HMR(Bottleneck, [3, 4, 6, 3], SMPL_MEAN_PARAMS)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

In [ ]:
# ViT as feature extractor (remove classification head)
vit_model = vit_b_16(weights=vit_weights)
vit_model.heads = nn.Identity()
vit_model.eval().to(device)

# Linear adapter: 768 → 2048 (untrained — random init)
adapter = nn.Linear(768, 2048).to(device)

In [ ]:
import json, zipfile
from torchvision import transforms as T


def run_regressor(features_2048, spin_model):
    """Run the SPIN regressor head on 2048-d features (iterative regression)."""
    B = features_2048.shape[0]
    pred_pose  = spin_model.init_pose.expand(B, -1)
    pred_shape = spin_model.init_shape.expand(B, -1)
    pred_cam   = spin_model.init_cam.expand(B, -1)
    for _ in range(3):
        xc = torch.cat([features_2048, pred_pose, pred_shape, pred_cam], 1)
        xc = spin_model.fc1(xc)
        xc = spin_model.drop1(xc)
        xc = spin_model.fc2(xc)
        xc = spin_model.drop2(xc)
        pred_pose  = spin_model.decpose(xc)  + pred_pose
        pred_shape = spin_model.decshape(xc) + pred_shape
        pred_cam   = spin_model.deccam(xc)   + pred_cam
    return pred_pose, pred_shape, pred_cam


# ══════════════════════════════════════════════════════════════
# Load Ground-Truth 3D annotations
# ══════════════════════════════════════════════════════════════

ANN_ZIP   = '/content/drive/MyDrive/dissertation/humans3.6m/annotations.zip'
SPIN_H36M = 'data/dataset_extras/h36m_train.npz'


def load_annotations():
    """Load H3.6M annotations — tries SPIN .npz first, then JSON from zip."""

    # ── Option A: SPIN-processed npz ──
    if os.path.exists(SPIN_H36M):
        print(f"Loading SPIN annotation file: {SPIN_H36M}")
        h36m = np.load(SPIN_H36M, allow_pickle=True)
        print(f"  Keys: {list(h36m.keys())}, Samples: {len(h36m['imgname'])}")
        gt_lookup = {str(name): i for i, name in enumerate(h36m['imgname'])}
        return dict(
            gt_lookup=gt_lookup,
            gt_j3d=h36m['S'],
            gt_center=h36m.get('center'),
            gt_scale=h36m.get('scale'),
        )

    # ── Option B: JSON annotations from zip ──
    print(f"Extracting {ANN_ZIP} ...")
    extract_dir = '/content/h36m_annotations'
    with zipfile.ZipFile(ANN_ZIP, 'r') as z:
        z.extractall(extract_dir)

    import glob as _glob
    json_files = sorted(_glob.glob(f'{extract_dir}/**/*.json', recursive=True))
    print(f"  Found {len(json_files)} JSON files")

    gt_lookup = {}    # image_name → index
    gt_j3d_list = []  # list of (J, 3) arrays
    gt_bbox_list = []  # list of [x, y, w, h] or None

    for jf in json_files:
        print(f"  Processing {os.path.basename(jf)} ...")
        with open(jf) as f:
            data = json.load(f)

        # ── COCO-style: {"images": [...], "annotations": [...]} ──
        if isinstance(data, dict) and 'annotations' in data:
            id2name = {}
            if 'images' in data:
                id2name = {img['id']: img['file_name'] for img in data['images']}

            for ann in data['annotations']:
                # Get image name
                img_name = id2name.get(ann.get('image_id'), ann.get('image_id', ''))
                if not img_name:
                    continue

                # Get 3D keypoints
                kp3d = ann.get('keypoints_3d') or ann.get('joints_3d') or ann.get('S')
                if kp3d is None:
                    continue
                kp3d = np.array(kp3d, dtype=np.float32)
                if kp3d.ndim == 1:
                    # Flat format: [x1,y1,z1, x2,y2,z2, ...] or [x1,y1,z1,c1, ...]
                    n_cols = 4 if len(kp3d) % 4 == 0 and len(kp3d) % 3 != 0 else 3
                    kp3d = kp3d.reshape(-1, n_cols)
                if kp3d.shape[-1] == 4:
                    kp3d = kp3d[:, :3]

                idx = len(gt_j3d_list)
                gt_lookup[str(img_name)] = idx
                gt_j3d_list.append(kp3d)

                bbox = ann.get('bbox')
                gt_bbox_list.append(bbox)

        # ── Flat list of dicts: [{"image": ..., "joints_3d": ...}, ...] ──
        elif isinstance(data, list):
            for entry in data:
                img_name = entry.get('image') or entry.get('file_name') or entry.get('imgname')
                kp3d = entry.get('keypoints_3d') or entry.get('joints_3d') or entry.get('S')
                if img_name is None or kp3d is None:
                    continue
                kp3d = np.array(kp3d, dtype=np.float32)
                if kp3d.ndim == 1:
                    n_cols = 4 if len(kp3d) % 4 == 0 and len(kp3d) % 3 != 0 else 3
                    kp3d = kp3d.reshape(-1, n_cols)
                if kp3d.shape[-1] == 4:
                    kp3d = kp3d[:, :3]

                idx = len(gt_j3d_list)
                gt_lookup[str(img_name)] = idx
                gt_j3d_list.append(kp3d)

                bbox = entry.get('bbox')
                gt_bbox_list.append(bbox)

    if not gt_j3d_list:
        print("  ⚠ Could not parse 3D joints from JSON. Check the exploration cell above")
        print("    and adjust the key names in this parser.")
        return dict(gt_lookup={}, gt_j3d=None, gt_center=None, gt_scale=None)

    gt_j3d = np.stack(gt_j3d_list)
    print(f"  Loaded {len(gt_lookup)} annotations, joints shape: {gt_j3d.shape}")

    # Convert bbox [x,y,w,h] → center/scale (SPIN-style)
    has_bbox = any(b is not None for b in gt_bbox_list)
    gt_center, gt_scale = None, None
    if has_bbox:
        centers, scales = [], []
        for bbox in gt_bbox_list:
            if bbox is not None:
                x, y, w, h = bbox[:4]
                centers.append([x + w/2, y + h/2])
                scales.append(max(w, h) / 200.0)
            else:
                centers.append([0, 0])
                scales.append(1.0)
        gt_center = np.array(centers, dtype=np.float32)
        gt_scale  = np.array(scales, dtype=np.float32)
        print(f"  Bbox info available → crop enabled")

    return dict(gt_lookup=gt_lookup, gt_j3d=gt_j3d, gt_center=gt_center, gt_scale=gt_scale)


ann = load_annotations()
gt_lookup = ann['gt_lookup']
GT_J3D    = ann['gt_j3d']
GT_CTR    = ann['gt_center']
GT_SCL    = ann['gt_scale']
HAS_CROP  = GT_CTR is not None and GT_SCL is not None

print(f"\nGT annotations: {len(gt_lookup)} images, crop info: {HAS_CROP}")
if GT_J3D is not None:
    print(f"Joints shape: {GT_J3D.shape}")
    print(f"Sample image names: {list(gt_lookup.keys())[:3]}")


# ══════════════════════════════════════════════════════════════
# Supervised Dataset — stream tar, match with GT 3D joints
# ══════════════════════════════════════════════════════════════

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def crop_person(img_pil, center, scale):
    """Crop person ROI using center + scale."""
    s = float(scale) * 200
    cx, cy = float(center[0]), float(center[1])
    half = s / 2
    x1, y1 = int(max(0, cx - half)), int(max(0, cy - half))
    x2, y2 = int(min(img_pil.width, cx + half)), int(min(img_pil.height, cy + half))
    return img_pil.crop((x1, y1, x2, y2))


class H36MSupervisedDataset(IterableDataset):
    """Stream images from tar archives, pair with GT 3D joints."""
    def __init__(self, archive_paths, gt_lookup, gt_joints,
                 gt_center=None, gt_scale=None, transform=None):
        self.archive_paths = archive_paths if isinstance(archive_paths, list) else [archive_paths]
        self.gt_lookup = gt_lookup
        self.gt_joints = gt_joints
        self.gt_center = gt_center
        self.gt_scale  = gt_scale
        self.transform = transform
        self.do_crop   = gt_center is not None and gt_scale is not None

    def _match_name(self, tar_name):
        """Try to match tar entry name to GT lookup (handles path prefixes)."""
        if tar_name in self.gt_lookup:
            return self.gt_lookup[tar_name]
        # Try just the filename
        basename = os.path.basename(tar_name)
        if basename in self.gt_lookup:
            return self.gt_lookup[basename]
        # Try relative path without leading dirs
        parts = tar_name.replace('\\', '/').split('/')
        for i in range(len(parts)):
            candidate = '/'.join(parts[i:])
            if candidate in self.gt_lookup:
                return self.gt_lookup[candidate]
        return None

    def __iter__(self):
        matched, skipped = 0, 0
        for path in self.archive_paths:
            print(f"  Streaming {os.path.basename(path)} ...")
            with tarfile.open(path, mode='r') as tar:
                for member in tar:
                    if not member.isfile():
                        continue
                    if not member.name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        continue
                    idx = self._match_name(member.name)
                    if idx is None:
                        skipped += 1
                        continue
                    try:
                        f = tar.extractfile(member)
                        img = Image.open(io.BytesIO(f.read())).convert('RGB')
                        if self.do_crop:
                            img = crop_person(img, self.gt_center[idx], self.gt_scale[idx])
                        if self.transform:
                            img = self.transform(img)
                        j3d = self.gt_joints[idx]
                        if j3d.shape[-1] == 4:
                            j3d = j3d[:, :3]
                        j3d = torch.tensor(j3d, dtype=torch.float32)
                        yield img, j3d
                        matched += 1
                    except Exception:
                        continue
        print(f"  Epoch done — matched: {matched}, skipped (no GT): {skipped}")


# ══════════════════════════════════════════════════════════════
# Training setup
# ══════════════════════════════════════════════════════════════

# Freeze SPIN regressor head
for name, param in spin_model.named_parameters():
    if any(k in name for k in ['fc1', 'fc2', 'decpose', 'decshape', 'deccam',
                                 'init_pose', 'init_shape', 'init_cam']):
        param.requires_grad = False

# Trainable: ViT + adapter
vit_model.train()
adapter = nn.Linear(768, 2048).to(device)

optimizer = torch.optim.AdamW(
    list(vit_model.parameters()) + list(adapter.parameters()),
    lr=1e-4, weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# Dataloader (streams all archives, yields only images with GT)
train_loader = DataLoader(
    H36MSupervisedDataset(
        ARCHIVES, gt_lookup, GT_J3D,
        gt_center=GT_CTR if HAS_CROP else None,
        gt_scale=GT_SCL if HAS_CROP else None,
        transform=train_transform,
    ),
    batch_size=16, num_workers=0,
)

NUM_EPOCHS = 10
LOG_EVERY  = 50
SAVE_PATH  = '/content/drive/MyDrive/dissertation/vit_adapter_supervised_ckpt.pt'

print(f"\nTraining ViT + adapter for {NUM_EPOCHS} epochs (supervised, MPJPE loss)")
print(f"GT images: {len(gt_lookup)}, Crop: {HAS_CROP}")
print(f"Trainable params: "
      f"{sum(p.numel() for p in vit_model.parameters() if p.requires_grad) + sum(p.numel() for p in adapter.parameters()):,}")

In [ ]:
history = {'epoch': [], 'loss': []}

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss, n_batches = 0.0, 0

    for step, (imgs, gt_j3d) in enumerate(train_loader, 1):
        imgs   = imgs.to(device)       # (B, 3, 224, 224)
        gt_j3d = gt_j3d.to(device)     # (B, J, 3)

        # Forward: ViT → adapter → SPIN regressor → SMPL → 3D joints
        feats    = vit_model(imgs)                                # (B, 768)
        feats_2k = adapter(feats)                                 # (B, 2048)
        pred_pose, pred_shape, pred_cam = run_regressor(feats_2k, spin_model)
        pred_rotmat = rot6d_to_rotmat(pred_pose).view(-1, 24, 3, 3)

        smpl_out = smpl_layer(
            global_orient=pred_rotmat[:, 0].unsqueeze(1),
            body_pose=pred_rotmat[:, 1:],
            betas=pred_shape,
            pose2rot=False,
        )
        pred_j3d = smpl_out.joints[:, :gt_j3d.shape[1]]          # match GT joint count

        # Root-relative alignment (subtract pelvis)
        pred_j3d_rel = pred_j3d - pred_j3d[:, [0]]
        gt_j3d_rel   = gt_j3d  - gt_j3d[:, [0]]

        # MPJPE loss (differentiable)
        loss = torch.sqrt(((pred_j3d_rel - gt_j3d_rel) ** 2).sum(-1) + 1e-8).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches  += 1

        if step % LOG_EVERY == 0:
            print(f"  epoch {epoch} | step {step:>5d} | MPJPE {loss.item()*1000:.1f} mm")

    scheduler.step()
    avg = epoch_loss / max(n_batches, 1)
    history['epoch'].append(epoch)
    history['loss'].append(avg)
    print(f"Epoch {epoch}/{NUM_EPOCHS}  avg_MPJPE={avg*1000:.1f} mm  "
          f"lr={scheduler.get_last_lr()[0]:.2e}")

# Save checkpoint
torch.save({
    'vit_state_dict':     vit_model.state_dict(),
    'adapter_state_dict': adapter.state_dict(),
    'optimizer':          optimizer.state_dict(),
    'history':            history,
}, SAVE_PATH)
print(f"\nCheckpoint saved → {SAVE_PATH}")

In [ ]:
# --- Training curve ---
plt.figure(figsize=(8, 4))
plt.plot(history['epoch'], [l * 1000 for l in history['loss']], marker='o')
plt.xlabel('Epoch'); plt.ylabel('MPJPE (mm)'); plt.title('Supervised Training — MPJPE')
plt.grid(True); plt.tight_layout(); plt.show()

# --- Re-evaluate on the quick-check batch from earlier ---
vit_model.eval()
with torch.no_grad():
    feats     = vit_model(images)
    feats_2k  = adapter(feats)
    ft_pose, ft_shape, ft_cam = run_regressor(feats_2k, spin_model)
    ft_rotmat = rot6d_to_rotmat(ft_pose).view(-1, 24, 3, 3)

out_ft = smpl_layer(
    global_orient=ft_rotmat[:, 0].unsqueeze(1),
    body_pose=ft_rotmat[:, 1:],
    betas=ft_shape,
    pose2rot=False,
)

# Compare: SPIN vs untrained hybrid vs supervised hybrid
mpjpe_untrained = torch.sqrt(((out_spin.joints - out_hybrid.joints) ** 2).sum(-1)).mean()
mpjpe_trained   = torch.sqrt(((out_spin.joints - out_ft.joints)     ** 2).sum(-1)).mean()

print(f"MPJPE (untrained hybrid vs SPIN): {mpjpe_untrained.item()*1000:.2f} mm")
print(f"MPJPE (supervised hybrid vs SPIN): {mpjpe_trained.item()*1000:.2f} mm")

# --- Side-by-side render ---
verts_ft = out_ft.vertices[0].detach().cpu().numpy()
trans_ft = cam_to_translation(ft_cam[0].detach().cpu().numpy())
render_ft = to_rgb_uint8(renderer(verts_ft, trans_ft, img_bgr.copy()))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_np);       axes[0].set_title("Input");                  axes[0].axis('off')
axes[1].imshow(render_spin);  axes[1].set_title("Standard SPIN");          axes[1].axis('off')
axes[2].imshow(render_ft);    axes[2].set_title("Supervised ViT + SPIN");  axes[2].axis('off')
plt.tight_layout(); plt.show()